# Sistema de Asistencia Inteligente - Unimarc
## Gestión de Inventario y Operaciones

In [3]:
import os
import time
import re
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# #1. Cargar variables de entorno
# Carga las credenciales y configuraciones desde el archivo .env para proteger datos sensibles
load_dotenv()

# #2. Configurar el modelo de lenguaje y la sesión de chat openai
# Se inicializa el modelo GPT-4o a través de la interfaz de ChatOpenAI
llm = ChatOpenAI(
    # Punto de enlace de la API (en este caso usando el proxy de GitHub Models)
    base_url=os.environ["OPENAI_BASE_URL"],

    # Token de autenticación obtenido de las variables de entorno
    api_key=os.environ["GITHUB_TOKEN"],

    # Modelo de última generación seleccionado para el procesamiento de lenguaje natural
    model="gpt-4o",

    # Grado de creatividad (0.7 permite respuestas fluidas y naturales sin perder coherencia)
    temperature=0.7,

    # Habilitación de streaming para recibir la respuesta fragmento a fragmento en tiempo real
    streaming=True,

    # Límite máximo de tokens para la generación de la respuesta
    max_tokens=300,
    
    # Configuración de muestreo por núcleo para mayor estabilidad en la calidad de salida
    top_p=1
)

In [4]:
# #3. Crear una estructura para almacenar el historial de conversaciones por sesión
# Se utiliza un diccionario para persistir la memoria de diferentes usuarios en la RAM del servidor
sesion_unimarc = {}

def historial_de_conversacion(sesion_id : str):
    # Si el ID de sesión no existe, se crea una nueva instancia de historial en memoria
    if sesion_id not in sesion_unimarc:
        sesion_unimarc[sesion_id] = InMemoryChatMessageHistory()
    return sesion_unimarc[sesion_id]


# #4. Función para extraer de manera efectiva información de inventario del texto del usuario, utilizando una técnica de ingeniería de prompts Zero-Shot.
def extraer_datos_inventario(texto_usuario):

    prompt_extraccion = (
        "Eres un experto en logística de Unimarc. "
        "Extrae los datos de inventario del siguiente texto y entrégalos."
        "EXCLUSIVAMENTE en formato JSON. No incluyas saludos ni explicaciones.\n\n"
        "Formato requerido:\n"
        "{\n"
        "  \"productos\": [\n"
        "    {\"nombre\": \"nombre del producto\", \"cantidad\": numero, \"unidad\": \"kg/unidades/cajas\"}\n"
        "  ]\n"
        "}\n\n"
        f"Texto: \"{texto_usuario}\"\n"
        "JSON:"
    )
    
    try:
        # Se asume que 'llm' ya está instanciado en tu entorno
        response = llm.invoke([HumanMessage(content=prompt_extraccion)])
        contenido = response.content.strip()
        
        # Limpieza por si el LLM devuelve el JSON dentro de bloques de código markdown
        if contenido.startswith("```json"):
            contenido = contenido.replace("```json", "").replace("```", "").strip()
        elif contenido.startswith("```"):
            contenido = contenido.replace("```", "").strip()

        # Esta es la línea clave completada: busca todo el bloque entre { y }
        match = re.search(r'\{.*\}', contenido, re.DOTALL)
        
        if match:
            json_extraido = match.group(0)
            # Convertimos el string JSON a un diccionario de Python
            datos_estructurados = json.loads(json_extraido)
            return datos_estructurados
        else:
            return {"error": "No se encontró formato JSON en la respuesta.", "raw": contenido}
            
    except json.JSONDecodeError as e:
        return {"error": f"Error al leer el JSON: {str(e)}", "raw": contenido}
    except Exception as e:
        return {"error": f"Ocurrió un error inesperado: {str(e)}"}

# #5. Función para sincronizar el contexto del historial de conversación, resumiendo si es necesario
def sincronizar_contexto_stock(sesion_id: str, max_mensajes=6):
    """
    Gestiona el límite de tokens mediante una estrategia de resumen.
    Cuando se supera el umbral, se comprimen los mensajes antiguos manteniendo 
    los datos críticos de inventario.
    """
    historial = historial_de_conversacion(sesion_id)

    if len(historial.messages) > max_mensajes:
        # Se separan los mensajes para resumir el pasado y mantener fresca la interacción inmediata
        mensajes_a_resumir = historial.messages[:-2]
        recientes = historial.messages[-2:]

        # Transformación del historial de objetos a texto plano para el procesamiento del LLM
        conversation_text = ""
        for msj in mensajes_a_resumir:
            role = "Usuario" if msj.type == "human" else "Asistente"
            conversation_text += f"{role}: {msj.content}\n"
        
        # Prompt de ingeniería de alta precisión para evitar pérdida de entidades (códigos/stock)
        prompt_resumen = (
            "Eres un experto en logística de Unimarc. Resume la siguiente conversación. "
            "REGLA DE ORO: No omitas códigos de productos, nombres de artículos ni cantidades de stock. "
            "Resume el resto en máximo 2 líneas de forma técnica.\n\n"
            f"Historial a resumir:\n{conversation_text}"
        )

        # Generación del resumen técnico optimizado en tokens
        summary_response = llm.invoke(prompt_resumen, max_tokens=150)
        summary = summary_response.content
        
        # Actualización del historial: Se limpia el exceso y se reinserta el resumen como contexto base
        historial.clear()
        historial.add_ai_message(f"[MEMORIA DE STOCK]: {summary}")
        historial.messages.extend(recientes)


# # NUEVO: BASE DE CONOCIMIENTO RAG (Protocolos Unimarc v2)
DOCUMENTOS_UNIMARC = [
    "LOGÍSTICA: El horario de recepción de proveedores en Unimarc es de 08:00 a 13:00 hrs.",
    "TEMPERATURA: Los productos perecibles deben ingresar con control estricto (máximo 4°C).",
    "MERMAS: Registrar código, cantidad y motivo (vencimiento/rotura) obligatoriamente.",
    "SEGURIDAD: El stock de seguridad mínimo para abarrotes es de 50 unidades por local.",
    "CATEGORÍAS: Pasillos válidos: Abarrotes, Frescos, Congelados, Perfumería y Limpieza.",
    "REGISTRO: Todo ingreso de stock debe indicar producto, cantidad y unidad de medida."
]

def recuperar_informacion(query):
    """Función RAG: Recupera reglas operativas según la consulta."""
    query_lower = query.lower()
    docs_relevantes = []
    
    for doc in DOCUMENTOS_UNIMARC:
        if any(word in doc.lower() for word in query_lower.split()):
            docs_relevantes.append(doc)
            
    return "\n".join(docs_relevantes[:2]) if docs_relevantes else "Usar criterio general de inventario."

# #6. configuración del prompt con RAG integrado y la función de conversación con memoria
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres la IA de Gestión de Inventario Unimarc. 
RESTRICCIÓN: Solo respondes sobre stock, mermas y logística.

REGLA CRÍTICA: Si el usuario se sale del tema de inventario, RESPONDE EXACTAMENTE: 
"ACCESO DENEGADO: Mi función está restringida exclusivamente a la gestión de inventario de Unimarc."

CONTEXTO RAG: {contexto_rag}

FORMATO: Sin emojis. Bloques de texto angostos. Salto de línea cada 8 palabras."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    historial_de_conversacion,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# #7. ejecutar_chat: para manejar la interacción, integrando RAG y sincronización de contexto
def ejecutar_chat(input_text, session_id):
    # 1. Recuperar contexto RAG
    contexto_recuperado = recuperar_informacion(input_text)
    
    # 2. Detección Zero-Shot para stock
    keywords = ["llegó", "anotar", "recibido", "ingresar", "entrada", "stock"]
    
    if any(p in input_text.lower() for p in keywords):
        print("[SISTEMA]: Detectado ingreso de inventario...")
        datos = extraer_datos_inventario(input_text)
        input_final = f"PROCESAR REGISTRO: {datos}"
    else:
        input_final = input_text

    sincronizar_contexto_stock(session_id)
    config = {"configurable": {"session_id": session_id}}
    
    try:
        # Retornamos la respuesta para que la función 8 la analice
        respuesta_completa = ""
        for chunk in conversation.stream({"input": input_final, "contexto_rag": contexto_recuperado}, config=config):
            respuesta_completa += chunk.content
            print(chunk.content, end="", flush=True)
        return respuesta_completa
    except Exception as e:
        return f"ERROR: {e}"

# #8. interfaz blindada para simular la interacción con el sistema de gestión de inventario
print("SISTEMA DE GESTIÓN DE INVENTARIO - UNIMARC")
print("MONITOREO DE SEGURIDAD OPERATIVO ACTIVADO\n")

id_sesion_actual = "admin_test_01"
contador_advertencias = 0 

while True:
    pregunta = input("Usuario: ")
    
    if pregunta.lower() in ["salir", "exit", "quit"]:
        print("SESIÓN FINALIZADA POR EL USUARIO.")
        break
        
    if pregunta.strip() == "":
        continue
    
    print(f"[INPUT]: {pregunta}")
    print(f"[OUTPUT]: ", end="", flush=True)
    
    respuesta_acumulada = ""
    try:
        config = {"configurable": {"session_id": id_sesion_actual}}
        contexto_rag = recuperar_informacion(pregunta)
        
        for chunk in conversation.stream(
            {"input": pregunta, "contexto_rag": contexto_rag}, 
            config=config
        ):
            texto_chunk = chunk.content
            print(texto_chunk, end="", flush=True)
            respuesta_acumulada += texto_chunk
        print("\n")
        
        # VERIFICACIÓN DE SEGURIDAD Y BLOQUEO POR REPETICIÓN
        if "ACCESO DENEGADO" in respuesta_acumulada:
            contador_advertencias += 1
            intentos_restantes = 3 - contador_advertencias
            
            if contador_advertencias < 3:
                print(f"ADVERTENCIA DE SEGURIDAD {contador_advertencias}/3: ACTIVIDAD NO AUTORIZADA.")
                print(f"INTENTOS RESTANTES ANTES DEL BLOQUEO DE SESIÓN: {intentos_restantes}\n")
            else:
                print("SISTEMA: SE HA ALCANZADO EL LÍMITE DE ADVERTENCIAS PERMITIDAS.")
                print("SISTEMA: CIERRE DE EMERGENCIA EJECUTADO POR PROTOCOLO DE SEGURIDAD.")
                break 
            
    except Exception as e:
        print(f"ERROR CRÍTICO DEL SISTEMA: {e}")
        break

SISTEMA DE GESTIÓN DE INVENTARIO - UNIMARC
MONITOREO DE SEGURIDAD OPERATIVO ACTIVADO

[INPUT]: hola
[OUTPUT]: ACCESO DENEGADO: Mi función está restringida exclusivamente  
a la gestión de inventario de Unimarc.

ADVERTENCIA DE SEGURIDAD 1/3: ACTIVIDAD NO AUTORIZADA.
INTENTOS RESTANTES ANTES DEL BLOQUEO DE SESIÓN: 2

[INPUT]: hablemos de la otganizacion del inventario para Santa Isabel
[OUTPUT]: ACCESO DENEGADO: Mi función está restringida exclusivamente  
a la gestión de inventario de Unimarc.

ADVERTENCIA DE SEGURIDAD 2/3: ACTIVIDAD NO AUTORIZADA.
INTENTOS RESTANTES ANTES DEL BLOQUEO DE SESIÓN: 1

[INPUT]: hablemos de inventarip
[OUTPUT]: Por favor especifica si deseas información sobre  
stock, mermas o logística para poder asistirte  
correctamente.

SESIÓN FINALIZADA POR EL USUARIO.
